In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

# ==========================================================
# 1. Configuration et chargement des modèles
# ==========================================================
BASE_MODEL = "facebook/nllb-200-distilled-600M"
# Assurez-vous que ce dossier existe là où votre notebook est lancé
ADAPTER    = "nllb-darija-lora-model"   

# Déterminer le device (GPU si disponible, sinon CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {device}")

# Charger le tokenizer et le modèle de base
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

# Appliquer l'adaptateur LoRA au modèle de base
model = PeftModel.from_pretrained(base_model, ADAPTER)

# Déplacer le modèle sur le bon device et le mettre en mode évaluation
model.to(device)
model.eval()
print("Modèle chargé et prêt pour l'inférence.")

# ==========================================================
# 2. Processus de traduction MANUEL et EXPLICITE
# ==========================================================
text_fr = "je veux manger"
src_lang = "fra_Latn"  # Code Hugging Face pour Français (Latin)
tgt_lang = "ary_Arab"  # Code pour Darija (Arabe)

print(f"\nTraduction de : '{text_fr}'")

# Étape 1 : Configurer le tokenizer avec la langue source
tokenizer.src_lang = src_lang

# Étape 2 : Tokeniser le texte d'entrée et l'envoyer sur le device
inputs = tokenizer(text_fr, return_tensors="pt").to(device)

# Étape 3 : Définir le token de début pour la langue cible (étape cruciale)
forced_bos_token_id = tokenizer.convert_tokens_to_ids(tgt_lang)

# Étape 4 : Lancer la génération
# 'with torch.no_grad()' est une bonne pratique pour l'inférence (plus rapide)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        forced_bos_token_id=forced_bos_token_id,
        max_new_tokens=100
    )

# Étape 5 : Décoder le résultat
result = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

print("👉 Traduction :", result)

Utilisation du device : cuda
Modèle chargé et prêt pour l'inférence.

Traduction de : 'je veux manger'
👉 Traduction : بغيت ناكول


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from peft import PeftModel
import torch

# 1) Identifiants
BASE_MODEL = "facebook/nllb-200-distilled-600M"
ADAPTER    = "nllb-darija-lora-model"   # ou ton chemin local vers l’adapter

# 2) Tokenizer et modèle de base
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

# 3) Charger l’adapter LoRA
model = PeftModel.from_pretrained(base_model, ADAPTER)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# 4) Créer la pipeline
translator = pipeline(
    "translation", 
    model=model, 
    tokenizer=tokenizer,
    src_lang="fra_Latn",       # code Hugging Face pour Français Latin
    tgt_lang="ary_Arab",       # code pour Darija Arab
    device=0 if device=="cuda" else -1
)

# 5) Traduire
text_fr = "je veux manger"
result = translator(text_fr)
print("👉 Traduction :", result[0]["translation_text"])


In [7]:
from azure.cognitiveservices.speech import SpeechConfig, SpeechSynthesizer

# Configuration avec votre clé Azure (créez-la sur portal.azure.com)
speech_config = SpeechConfig(
    subscription="votre-clé-api-ici",  # Ex: "a1b2c3d4e5f6g7h8i9j0"
    region="eastus"                   # "eastus" ou "westeurope"
)

# Voix marocaine native (choisissez l'une ou l'autre)
speech_config.speech_synthesis_voice_name = "ar-MA-JamalNeural"  # Masculin
# speech_config.speech_synthesis_voice_name = "ar-MA-MounaNeural"  # Féminin

# Synthèse du texte
synthesizer = SpeechSynthesizer(speech_config)
result = synthesizer.speak_text_async("بغيت ناكول كسكس ف مرّاكش").get()

# Sauvegarde du fichier audio
with open("darija.wav", "wb") as audio_file:
    audio_file.write(result.audio_data)

print("✅ Fichier audio généré : darija.wav")

ALSA lib confmisc.c:855:(parse_card) cannot find card '0'
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_card_inum returned error: No such file or directory
ALSA lib confmisc.c:422:(snd_func_concat) error evaluating strings
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_concat returned error: No such file or directory
ALSA lib confmisc.c:1342:(snd_func_refer) error evaluating name
ALSA lib conf.c:5204:(_snd_config_evaluate) function snd_func_refer returned error: No such file or directory
ALSA lib conf.c:5727:(snd_config_expand) Evaluate error: No such file or directory
ALSA lib pcm.c:2721:(snd_pcm_open_noupdate) Unknown PCM default


✅ Fichier audio généré : darija.wav
